# 하이브리드 파이프라인 4단계 Ablation (E3)

`embedding_model_comparison.ipynb` / `reranker_model_comparison.ipynb`에서 고른 1위
모델들을 가지고, 파이프라인을 단계별로 켜고 끄면서 Recall@K/MRR이 어떻게 변하는지 본다.

| 단계 | 구성 | 확인하려는 것 |
|---|---|---|
| A. 키워드만 | `keyword_search` | 규칙 기반 매칭의 baseline 성능 |
| B. 임베딩만 | `embedding_search`(1위 모델) | 의미 기반 검색 단독 성능 |
| C1. 머지(현재 방식) | 키워드∪임베딩, 키워드 후보 우선 정렬 | 지금 `reranker.py`의 identity 폴백과 동일한 규칙 — 리랭커 없이 두 결과를 합치기만 하면 얼마나 나오는지 |
| C2. 머지(RRF) | 키워드∪임베딩, RRF로 순위 융합 | "키워드 우선 이진 규칙" 대신 순위 기반 융합을 쓰면 달라지는지 |
| D. 전체 하이브리드+리랭크 | C1/C2 후보 풀 + 1위 리랭커 | 리랭커가 최종적으로 얼마나 더 기여하는지 |

⚠️ 주의: "키워드 후보가 항상 임베딩-only 후보보다 우선"이라는 규칙은 `reranker.py`의
**리랭커가 죽었을 때(identity 폴백)만** 적용되는 동작이다. 실제 리랭커가 살아있는
D 단계에서는 모든 후보를 리랭커가 동일하게 재채점하므로 이 규칙이 적용되지 않는다 —
그래서 C1 단계는 "리랭커 없이 이 규칙만 썼을 때"를 따로 측정하는 것이다.

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd()
if not (project_root / "agent").exists():
    for parent in Path.cwd().parents:
        if (parent / "agent").exists():
            project_root = parent
            break
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("project_root:", project_root)

In [ ]:
import gc
import json

import numpy as np
import torch
from sentence_transformers import SentenceTransformer

from agent.mapping.golden_set import load_golden_set
from agent.mapping.eval_metrics import evaluate, summarize, rrf_merge
from agent.mapping.keyword_search import keyword_search
from agent.mapping.reranker import _merge_candidates
from agent.interfaces import Claim, TableCandidate

examples = load_golden_set()
print(f"평가 가능 claim: {len(examples)}건")

In [ ]:
# 앞 두 노트북 결과를 확인한 뒤 아래 3개 상수를 갱신하세요.
WINNING_EMBEDDING_MODEL = "Qwen/Qwen3-Embedding-4B"
WINNING_RERANKER_MODEL = "Qwen/Qwen3-Reranker-4B"
WINNING_RERANKER_KIND = "qwen"  # "qwen"(생성형 yes/no) 또는 "cross_encoder"(BGE/GTE)

QWEN_QUERY_INSTRUCTION = (
    "Given a Korean news claim sentence, retrieve the KOSIS statistical table "
    "description that best matches it"
)

catalog = json.loads((project_root / "agent" / "mapping" / "table_catalog.json").read_text(encoding="utf-8"))
catalog_ids = [t["tblId"] for t in catalog["tables"]]
catalog_texts = [t["embedding_text"] for t in catalog["tables"]]
doc_text_map = {t["tblId"]: t["embedding_text"] for t in catalog["tables"]}

embed_model = SentenceTransformer(WINNING_EMBEDDING_MODEL)
_doc_texts = catalog_texts if "multilingual-e5" not in WINNING_EMBEDDING_MODEL else [f"passage: {t}" for t in catalog_texts]
_doc_vecs = np.array(embed_model.encode(_doc_texts, convert_to_numpy=True, show_progress_bar=False))


def embedding_ranking(sentence: str, top_k: int = 10) -> list[tuple[str, float]]:
    if "Qwen3-Embedding" in WINNING_EMBEDDING_MODEL:
        q_text = f"Instruct: {QWEN_QUERY_INSTRUCTION}\nQuery: {sentence}"
    elif "multilingual-e5" in WINNING_EMBEDDING_MODEL:
        q_text = f"query: {sentence}"
    else:
        q_text = sentence
    q_vec = np.array(embed_model.encode([q_text], convert_to_numpy=True, show_progress_bar=False)[0])
    q = q_vec / (np.linalg.norm(q_vec) + 1e-8)
    d = _doc_vecs / (np.linalg.norm(_doc_vecs, axis=1, keepdims=True) + 1e-8)
    sims = d @ q
    order = np.argsort(-sims)[:top_k]
    return [(catalog_ids[i], float(sims[i])) for i in order]

## 단계 A/B: 키워드만 / 임베딩만

In [ ]:
def stage_a_keyword_only(sentence: str, top_k: int = 10) -> list[str]:
    claim = Claim(sentence=sentence, claim_type="규모")
    return [c.table_id for c in keyword_search(claim, top_k=top_k)]


def stage_b_embedding_only(sentence: str, top_k: int = 10) -> list[str]:
    return [tid for tid, _ in embedding_ranking(sentence, top_k=top_k)]


result_a = evaluate("A. 키워드만", stage_a_keyword_only, examples)
result_b = evaluate("B. 임베딩만", stage_b_embedding_only, examples)
print(f"A: Recall@5={result_a.recall_at_k[5]:.1%}  MRR={result_a.mrr:.3f}")
print(f"B: Recall@5={result_b.recall_at_k[5]:.1%}  MRR={result_b.mrr:.3f}")

## 단계 C1/C2: 머지 (리랭커 없이 후보만 합침)

In [ ]:
def _merged_pool(sentence: str, top_k: int = 10) -> list[TableCandidate]:
    claim = Claim(sentence=sentence, claim_type="규모")
    kw_candidates = keyword_search(claim, top_k=top_k)
    emb_candidates = [
        TableCandidate(
            table_id=tid, table_name=doc_text_map.get(tid, tid)[:30], score=score,
            source_meta=f"embedding_search model={WINNING_EMBEDDING_MODEL}",
        )
        for tid, score in embedding_ranking(sentence, top_k=top_k)
    ]
    return _merge_candidates(kw_candidates, emb_candidates)


def stage_c1_merge_keyword_priority(sentence: str, top_k: int = 5) -> list[str]:
    """reranker.py의 identity 폴백과 동일한 규칙: 키워드 검증된 후보 우선, 그 안에서 점수 내림차순."""
    pool = _merged_pool(sentence)

    def sort_key(c: TableCandidate):
        unverified = "(embedding-only, unverified)" in (c.source_meta or "")
        return (unverified, -c.score)

    return [c.table_id for c in sorted(pool, key=sort_key)][:top_k]


def stage_c2_merge_rrf(sentence: str, top_k: int = 5) -> list[str]:
    """키워드 우선 규칙 대신 RRF로 두 순위를 융합."""
    kw_ranking = stage_a_keyword_only(sentence, top_k=10)
    emb_ranking = stage_b_embedding_only(sentence, top_k=10)
    return rrf_merge([kw_ranking, emb_ranking])[:top_k]


result_c1 = evaluate("C1. 머지(키워드 우선)", stage_c1_merge_keyword_priority, examples)
result_c2 = evaluate("C2. 머지(RRF)", stage_c2_merge_rrf, examples)
print(f"C1: Recall@5={result_c1.recall_at_k[5]:.1%}  MRR={result_c1.mrr:.3f}")
print(f"C2: Recall@5={result_c2.recall_at_k[5]:.1%}  MRR={result_c2.mrr:.3f}")

## 단계 D: 전체 하이브리드 + 리랭커

In [ ]:
RERANKER_INSTRUCTION = (
    "Given a Korean news claim sentence, judge whether the KOSIS statistical table "
    "description is the correct match"
)
_PREFIX = (
    "<|im_start|>system\n"
    "Judge whether the Document meets the requirements based on the Query and the Instruct "
    'provided. Note that the answer can only be "yes" or "no".<|im_end|>\n<|im_start|>user\n'
)
_SUFFIX = "<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n"
_MAX_LEN = 4096


def _load_qwen_reranker(model_name: str):
    from transformers import AutoModelForCausalLM, AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained(model_name, padding_side="left")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = AutoModelForCausalLM.from_pretrained(
        model_name, torch_dtype=torch.float16 if device == "cuda" else torch.float32
    ).to(device).eval()
    prefix_ids = tokenizer.encode(_PREFIX, add_special_tokens=False)
    suffix_ids = tokenizer.encode(_SUFFIX, add_special_tokens=False)
    true_id = tokenizer.convert_tokens_to_ids("yes")
    false_id = tokenizer.convert_tokens_to_ids("no")

    def scorer(query: str, docs: list[str]) -> list[float]:
        if not docs:
            return []
        pairs = [f"<Instruct>: {RERANKER_INSTRUCTION}\n<Query>: {query}\n<Document>: {doc}" for doc in docs]
        scores: list[float] = []
        with torch.no_grad():
            for i in range(0, len(pairs), 8):
                batch = pairs[i:i + 8]
                inputs = tokenizer(
                    batch, padding=False, truncation="longest_first",
                    return_attention_mask=False, max_length=_MAX_LEN - len(prefix_ids) - len(suffix_ids),
                )
                for j, ids in enumerate(inputs["input_ids"]):
                    inputs["input_ids"][j] = prefix_ids + ids + suffix_ids
                inputs = tokenizer.pad(inputs, padding=True, return_tensors="pt", max_length=_MAX_LEN)
                inputs = {k: v.to(device) for k, v in inputs.items()}
                logits = model(**inputs).logits[:, -1, :]
                stacked = torch.stack([logits[:, false_id], logits[:, true_id]], dim=1)
                probs = torch.nn.functional.log_softmax(stacked, dim=1)
                scores.extend(probs[:, 1].exp().tolist())
        return scores

    return scorer, model


def _load_cross_encoder_reranker(model_name: str):
    from sentence_transformers import CrossEncoder

    model = CrossEncoder(model_name, trust_remote_code=True)

    def scorer(query: str, docs: list[str]) -> list[float]:
        if not docs:
            return []
        return model.predict([(query, doc) for doc in docs]).tolist()

    return scorer, model


if WINNING_RERANKER_KIND == "qwen":
    _rerank_scorer, _reranker_model = _load_qwen_reranker(WINNING_RERANKER_MODEL)
else:
    _rerank_scorer, _reranker_model = _load_cross_encoder_reranker(WINNING_RERANKER_MODEL)


def stage_d_full_hybrid_rerank(sentence: str, top_k: int = 5) -> list[str]:
    pool = _merged_pool(sentence)
    if not pool:
        return []
    docs = [doc_text_map.get(c.table_id, c.table_name) for c in pool]
    scores = _rerank_scorer(sentence, docs)
    ranked = sorted(zip(pool, scores), key=lambda pair: pair[1], reverse=True)
    return [c.table_id for c, _ in ranked][:top_k]


result_d = evaluate("D. 전체 하이브리드+리랭크", stage_d_full_hybrid_rerank, examples)
print(f"D: Recall@5={result_d.recall_at_k[5]:.1%}  MRR={result_d.mrr:.3f}")

del _reranker_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 종합 비교

In [ ]:
all_results = [result_a, result_b, result_c1, result_c2, result_d]
summarize(all_results)

## 발표에서 쓸 포인트

- **A→B**: 키워드 매칭이 못 잡는 동의어/의역 표현을 임베딩이 얼마나 보완하는지.
- **B→C1**: 두 방법을 그냥 합치기만 해도(리랭커 없이) 단일 방법 대비 얼마나 오르는지.
- **C1 vs C2**: "키워드 후보 우선"이라는 지금의 이진 규칙이 RRF 같은 순위 융합보다
  나은지 — 여기서 지면 지금 설계를 순위 융합으로 바꿀 근거가 되고, 이기면 지금 설계를
  유지할 근거가 됨.
- **C1/C2→D**: 무거운 리랭커 모델을 추가로 돌리는 게 정말 그만한 값을 하는지(delta가
  작으면 "리랭커 없이도 충분하다"는 비용 절감 주장의 근거가 됨).

표본 40건(그중 16건 low-confidence)이라는 점은 발표에서 그대로 밝힐 것.